# NIT Andhra Pradesh — Dept. of ECE
## ECM-402: AI for Signal Processing Applications — Lab 5 (Sequential Networks: RNN and LSTM)
### Part II: Speech recognition (keyword spotting) using RNN

**Name:** Vivekanand Ojha  |  **Roll No:** _<your roll number>_  |  **Instructor:** Dr. V. Prakash Singh

## Problem statement
Build an RNN keyword-spotting classifier for 5 keywords (**one, two, three, right, left**) from the `speech_commands_small` folder:
MFCC (13 coeff., 25 ms frame, 10 ms hop, 16 kHz) → RNN (hidden 64, 1 layer) → Linear(64, 5) → softmax.
Split 70/15/15, train 10 epochs (Adam, lr = 1e-3, batch 32, cross-entropy), plot train/val accuracy, report test accuracy and confusion matrix.

$h_t=\tanh(W_{xh}x_t+W_{hh}h_{t-1}+b_h)$, $\hat y=\text{softmax}(W_{hy}h_T+b_y)$, $L=-\sum_i y_i\log\hat y_i$

**Setup:** `pip install torch librosa scikit-learn matplotlib scipy` and place the `speech_commands_small/` folder next to this notebook.

In [ ]:
import os, glob, random, warnings
import numpy as np
import matplotlib.pyplot as plt
import librosa, librosa.display
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

DATA_DIR = "speech_commands_small"            # <-- folder with 5 sub-folders: one, two, three, right, left
CLASSES  = ["one", "two", "three", "right", "left"]
SR, N_FFT, HOP = 16000, 400, 160              # 16 kHz, 25 ms frame (400 samples), 10 ms hop (160 samples)
N_MFCC, N_MELS = 13, 40
MAX_LEN = 101                                 # 1 s of audio -> 101 frames
os.makedirs("figures", exist_ok=True)

In [ ]:
# ---- OPTIONAL fallback: only runs if the real dataset folder is NOT found ----
# It generates a tiny synthetic dataset so the notebook can be tested end-to-end.
# On your lab machine put the provided folder next to this notebook and this cell does nothing.
if not os.path.isdir(DATA_DIR):
    from scipy.io import wavfile
    print("!! Dataset not found -> generating SYNTHETIC data for a dry run. Results are NOT meaningful.")
    rng = np.random.default_rng(0)
    for ci, c in enumerate(CLASSES):
        os.makedirs(os.path.join(DATA_DIR, c), exist_ok=True)
        for i in range(60):
            n = int(SR * rng.uniform(0.5, 1.0)); t = np.arange(n) / SR
            f0 = 300 + 150 * ci
            sig = np.sin(2*np.pi*(f0 + (100*(ci+1))*t) * t) * np.hanning(n) + 0.05 * rng.standard_normal(n)
            wavfile.write(os.path.join(DATA_DIR, c, f"{c}_{i}.wav"), SR, (sig * 12000).astype(np.int16))

## S1: Data analysis and visualization

In [ ]:
# ---------------- S1: data inspection ----------------
files, labels = [], []
for ci, c in enumerate(CLASSES):
    fs = sorted(glob.glob(os.path.join(DATA_DIR, c, "*.wav")))
    files += fs; labels += [ci] * len(fs)
labels = np.array(labels)

print("Samples per class:")
for ci, c in enumerate(CLASSES): print(f"  {c:6s}: {(labels == ci).sum()}")
print("Total:", len(files))

waves = [librosa.load(f, sr=SR, mono=True)[0] for f in files]
durations = np.array([len(w) / SR for w in waves])
rms = np.array([np.sqrt(np.mean(w ** 2)) for w in waves])
print(f"Duration (s): min={durations.min():.2f}, mean={durations.mean():.2f}, max={durations.max():.2f}")

fig, ax = plt.subplots(1, 3, figsize=(15, 3.5))
ax[0].bar(CLASSES, [(labels == i).sum() for i in range(5)]); ax[0].set_title("Samples per class")
ax[1].hist(durations, bins=20); ax[1].set_title("Duration distribution (s)")
ax[2].boxplot([rms[labels == i] for i in range(5)]); ax[2].set_xticklabels(CLASSES); ax[2].set_title("RMS energy per class")
plt.tight_layout(); plt.savefig("figures/data_analysis.png", dpi=150); plt.show()

# example log-mel spectrogram for each keyword
fig, axes = plt.subplots(1, 5, figsize=(20, 3.5))
for ci, c in enumerate(CLASSES):
    idx = np.where(labels == ci)[0][0]
    S = librosa.power_to_db(librosa.feature.melspectrogram(y=waves[idx], sr=SR, n_fft=N_FFT, hop_length=HOP, n_mels=N_MELS), ref=np.max)
    librosa.display.specshow(S, sr=SR, hop_length=HOP, x_axis="time", y_axis="mel", ax=axes[ci])
    axes[ci].set_title(f"'{c}'  ({durations[idx]:.2f}s)")
plt.tight_layout(); plt.savefig("figures/spectrograms.png", dpi=150); plt.show()

**Observation:** utterances differ in length (many are shorter than 1 s) and in loudness (RMS), which is why features are normalised and padded/truncated to a fixed length. Different keywords show different formant/energy patterns over time, which is what the RNN exploits.

## S2: Data splitting and feature extraction

In [ ]:
# ---------------- S2: 70 / 15 / 15 stratified split ----------------
idx_all = np.arange(len(files))
idx_tr, idx_tmp = train_test_split(idx_all, test_size=0.30, stratify=labels, random_state=SEED)
idx_va, idx_te  = train_test_split(idx_tmp, test_size=0.50, stratify=labels[idx_tmp], random_state=SEED)
print("Train / Val / Test:", len(idx_tr), len(idx_va), len(idx_te))

# ---------------- S2: feature extraction (13 MFCC, 25 ms window, 10 ms hop) ----------------
def mfcc_feat(w):
    m = librosa.feature.mfcc(y=w, sr=SR, n_mfcc=N_MFCC, n_fft=N_FFT, hop_length=HOP, n_mels=N_MELS)
    return m.T.astype(np.float32)                         # (frames, 13)

raw = [mfcc_feat(w) for w in waves]

# normalise with TRAINING-set statistics only
train_frames = np.concatenate([raw[i] for i in idx_tr], axis=0)
mu, sd = train_frames.mean(0), train_frames.std(0) + 1e-8

def pad_trunc(f, L=MAX_LEN):
    if len(f) >= L: return f[:L]
    return np.vstack([f, np.zeros((L - len(f), f.shape[1]), dtype=np.float32)])   # zero = training mean after normalisation

feats = np.stack([pad_trunc((f - mu) / sd) for f in raw])  # (N, T, D)
print("Feature tensor:", feats.shape, "(N, time steps, feature dim)")
D = feats.shape[2]

## S3: Data loading

In [ ]:
# ---------------- S3: Dataset + DataLoader ----------------
class KWSDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32); self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]

BATCH = 32
train_dl = DataLoader(KWSDataset(feats[idx_tr], labels[idx_tr]), batch_size=BATCH, shuffle=True)
val_dl   = DataLoader(KWSDataset(feats[idx_va], labels[idx_va]), batch_size=BATCH)
test_dl  = DataLoader(KWSDataset(feats[idx_te], labels[idx_te]), batch_size=BATCH)
xb, yb = next(iter(train_dl)); print("Batch shape:", xb.shape, yb.shape)

## S4: Model architecture (RNN, hidden = 64, 1 layer, Linear(64, 5))

In [ ]:
class RNNClassifier(nn.Module):
    def __init__(self, input_dim, hidden=64, n_classes=5):
        super().__init__()
        self.rnn = nn.RNN(input_dim, hidden, num_layers=1, nonlinearity="tanh", batch_first=True)
        self.fc  = nn.Linear(hidden, n_classes)
    def forward(self, x):                 # x: (batch, time, feat)
        _, h_n = self.rnn(x)              # h_n: (layers, batch, hidden)
        return self.fc(h_n[-1])           # logits; softmax is inside CrossEntropyLoss (applied explicitly at test time)

model = RNNClassifier(D)
print(model)
print("Trainable parameters:", sum(p.numel() for p in model.parameters()))

## S5: Training

In [ ]:
# ---------------- S5: training (10 epochs, Adam lr=1e-3, CrossEntropyLoss, batch 32) ----------------
EPOCHS = 10
model = model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

def run_epoch(dl, train):
    model.train(train)
    tot_loss = correct = n = 0
    with torch.set_grad_enabled(train):
        for xb, yb in dl:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)
            if train:
                optimizer.zero_grad(); loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 5.0)   # guards against exploding gradients
                optimizer.step()
            tot_loss += loss.item() * len(yb); correct += (logits.argmax(1) == yb).sum().item(); n += len(yb)
    return tot_loss / n, correct / n

hist = {"tr_loss": [], "tr_acc": [], "va_loss": [], "va_acc": []}
for ep in range(1, EPOCHS + 1):
    tl, ta = run_epoch(train_dl, True); vl, va = run_epoch(val_dl, False)
    for k, v in zip(hist, [tl, ta, vl, va]): hist[k].append(v)
    print(f"Epoch {ep:2d}/{EPOCHS} | train loss {tl:.4f} acc {ta:.4f} | val loss {vl:.4f} acc {va:.4f}")

## S6: Evaluation and visualization

In [ ]:
# ---------------- S6: accuracy curves ----------------
ep = range(1, EPOCHS + 1)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(ep, hist["tr_acc"], "o-", label="Train"); ax[0].plot(ep, hist["va_acc"], "s-", label="Validation")
ax[0].set_xlabel("Epoch"); ax[0].set_ylabel("Accuracy"); ax[0].set_title("RNN: accuracy vs epoch"); ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].plot(ep, hist["tr_loss"], "o-", label="Train"); ax[1].plot(ep, hist["va_loss"], "s-", label="Validation")
ax[1].set_xlabel("Epoch"); ax[1].set_ylabel("Loss"); ax[1].set_title("RNN: loss vs epoch"); ax[1].legend(); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.savefig("figures/rnn_training_curves.png", dpi=150); plt.show()

# ---------------- S6: test accuracy + confusion matrix ----------------
model.eval(); preds, trues = [], []
with torch.no_grad():
    for xb, yb in test_dl:
        probs = torch.softmax(model(xb.to(device)), dim=1)        # class probabilities
        preds += probs.argmax(1).cpu().tolist(); trues += yb.tolist()
preds, trues = np.array(preds), np.array(trues)
print(f"Final TEST accuracy: {(preds == trues).mean() * 100:.2f} %")

cm = confusion_matrix(trues, preds, labels=range(5))
disp = ConfusionMatrixDisplay(cm, display_labels=CLASSES)
fig, ax = plt.subplots(figsize=(6, 5)); disp.plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title("RNN: confusion matrix (test set)"); plt.tight_layout()
plt.savefig("figures/rnn_confusion_matrix.png", dpi=150); plt.show()
print(classification_report(trues, preds, target_names=CLASSES, digits=3))

### Discussion
* Compare final train vs validation accuracy to judge over-fitting.
* Read the confusion matrix row-wise: the diagonal is correct recognition; large off-diagonal entries show which keywords the model mixes up (acoustically similar words such as *two* / *three* are typical).
* A plain RNN on ~100 time steps suffers from vanishing gradients and may learn slowly — Part IV repeats the experiment with an LSTM for comparison.